In [1]:
import json
import time

import faiss
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer


/Users/aaravmenon/Aalen/database/project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1030 12:33:30.644000 64490 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [3]:
files = ["naruto_s1_backup.json", "naruto_s2.json", "naruto_s2_secondpart.json", "naruto_s2_thirdpart.json"]

documents = []
ids = []
metadata = []
full_dataset = []  # Added to track dictionaries for both files cleanly

for file in files:
    with open(file, "r", encoding="utf-8") as f:
        current_season = json.load(f)

    documents.extend([doc["raw_text"] for doc in current_season])
    ids.extend([str(doc["id"]) for doc in current_season])
    metadata.extend([{"url": doc["url"]} for doc in current_season])
    full_dataset.extend(current_season)  # Keep the dictionaries aligned with the index

print(f"Loaded {len(documents)} total documents into memory.")

Loaded 720 total documents into memory.


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

document_embeddings = model.encode(
    documents,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype("float32")

print("Embedding shape:", document_embeddings.shape)


Batches: 100%|██████████| 23/23 [00:08<00:00,  2.66it/s]

Embedding shape: (720, 384)


In [6]:
chroma_client = chromadb.PersistentClient(path="./chroma_db")

if "naruto_documents" in {
    collection.name for collection in chroma_client.list_collections()
}:
    chroma_client.delete_collection("naruto_documents")

chroma_collection = chroma_client.create_collection(
    name="naruto_documents",
    configuration={"hnsw": {"space": "cosine"}},
)

chroma_collection.add(
    ids=ids,
    embeddings=document_embeddings.tolist(),
    documents=documents,
    metadatas=metadata,
)

print(f"Stored {chroma_collection.count()} documents in ChromaDB")


Stored 720 documents in ChromaDB


In [7]:
faiss_index = faiss.IndexFlatIP(document_embeddings.shape[1])
faiss_index.add(document_embeddings)

print(f"Stored {faiss_index.ntotal} documents in FAISS")


Stored 720 documents in FAISS


In [8]:
def chroma_search(query, top_k=5):
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    result = chroma_collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    return [
        {
            "id": result["ids"][0][rank],
            "score": 1 - result["distances"][0][rank],
            "url": result["metadatas"][0][rank]["url"],
            "text": result["documents"][0][rank],
        }
        for rank in range(len(result["ids"][0]))
    ]


In [9]:
def faiss_search(query, top_k=5):
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, indices = faiss_index.search(query_embedding, top_k)

    return [
        {
            "id": ids[index],
            "score": float(score),
            "url": full_dataset[index]["url"],  # Fixed: Looks up the aligned dictionary list
            "text": full_dataset[index]["raw_text"],  # Fixed: Looks up the aligned dictionary list
        }
        for score, index in zip(scores[0], indices[0])
        if 0 <= index < len(ids)  # Double-sided bounds safety check
    ]

In [10]:
def print_results(name, results):
    print(f"\n{name}")
    for rank, result in enumerate(results, start=1):
        print(f"\n{rank}. Score: {result['score']:.4f}")
        print("URL:", result["url"])
        print("Text:", result["text"][:300])


In [12]:
print("FAISS vectors:", faiss_index.ntotal)
print("Dataset documents:", len(full_dataset))
print("IDs:", len(ids))

FAISS vectors: 720
Dataset documents: 720
IDs: 720


In [13]:
query = "Nine tail fox attack"

start = time.perf_counter()
chroma_results = chroma_search(query)
chroma_time = time.perf_counter() - start

start = time.perf_counter()
faiss_results = faiss_search(query)
faiss_time = time.perf_counter() - start

print_results("ChromaDB results", chroma_results)
print(f"ChromaDB search time: {chroma_time:.6f} seconds")

print_results("FAISS results", faiss_results)
print(f"FAISS search time: {faiss_time:.6f} seconds")



ChromaDB results

1. Score: 0.5059
URL: https://myanimelist.net/anime/1735/Naruto__Shippuuden/episode/41
Text: Synopsis Naruto gets more enraged in his three-tailed state, and throws Kabuto away, who inadvertently knocks Sakura out too. Naruto's anger eventually escalates to him destroying the Tenchi Bridge and he takes to a forest to battle Orochimaru. Sai creates an ink bird and flies beneath the collapsin

2. Score: 0.4855
URL: https://myanimelist.net/anime/1735/Naruto__Shippuuden/episode/432
Text: Synopsis Being caught in the Infinite Tsukuyomi, Tsunade dreams about changing the past that she sits near a lake and hears Jiraiya telling her that he finished his latest work before leaving. She looks at the papers, which are titled "Jiraiya's Ninja Scrolls" and recalls when he was still in the mi

3. Score: 0.4755
URL: https://myanimelist.net/anime/1735/Naruto__Shippuuden/episode/373
Text: Synopsis As the newly reformed Team 7 mobilizes to battle, the Ten Tails creates an army of mons

In [14]:
queries = [
    "Naruto fails an exam",
    "Sasuke fights Gaara",
    "Naruto learns a new technique",
    "a ninja team completes a dangerous mission",
    "Naruto protects his friends",
    "Sakura trains to become stronger",
    "Kakashi teaches Team 7",
    "a ninja passes an important test",
    "Sasuke leaves the Hidden Leaf Village",
    "Gaara uses his sand powers",
    "Naruto battles a dangerous enemy",
    "the Chunin Exams begin",
    "a character learns a new jutsu",
    "Team 7 goes on a mission",
    "a village is attacked",
    "Naruto and Sasuke fight together",
    "a ninja is injured during battle",
    "the characters prepare for a tournament",
    "someone tries to protect their family",
    "a powerful enemy is defeated",
]

def benchmark_search(search_function, queries, top_k=5):
    # Warm up model and database before measuring repeated searches.
    search_function(queries[0], top_k=top_k)
    timings = []
    results = {}

    for query in queries:
        start = time.perf_counter()
        results[query] = search_function(query, top_k=top_k)
        timings.append(time.perf_counter() - start)

    return results, np.asarray(timings)

chroma_benchmark_results, chroma_timings = benchmark_search(chroma_search, queries)
faiss_benchmark_results, faiss_timings = benchmark_search(faiss_search, queries)

print("Average search time")
print(f"ChromaDB: {chroma_timings.mean():.6f} seconds")
print(f"FAISS:    {faiss_timings.mean():.6f} seconds")
print()
print("Median search time")
print(f"ChromaDB: {np.median(chroma_timings):.6f} seconds")
print(f"FAISS:    {np.median(faiss_timings):.6f} seconds")

for query in queries:
    chroma_ids = [result["id"] for result in chroma_benchmark_results[query]]
    faiss_ids = [result["id"] for result in faiss_benchmark_results[query]]
    overlap = len(set(chroma_ids) & set(faiss_ids))
    print(f"\\n{query}")
    print(f"Top-5 result overlap: {overlap}/5")
    print(f"Same ranking: {chroma_ids == faiss_ids}")


Average search time
ChromaDB: 0.013302 seconds
FAISS:    0.011524 seconds

Median search time
ChromaDB: 0.012681 seconds
FAISS:    0.010922 seconds
\nNaruto fails an exam
Top-5 result overlap: 5/5
Same ranking: True
\nSasuke fights Gaara
Top-5 result overlap: 5/5
Same ranking: True
\nNaruto learns a new technique
Top-5 result overlap: 5/5
Same ranking: True
\na ninja team completes a dangerous mission
Top-5 result overlap: 5/5
Same ranking: True
\nNaruto protects his friends
Top-5 result overlap: 5/5
Same ranking: True
\nSakura trains to become stronger
Top-5 result overlap: 5/5
Same ranking: True
\nKakashi teaches Team 7
Top-5 result overlap: 5/5
Same ranking: True
\na ninja passes an important test
Top-5 result overlap: 5/5
Same ranking: True
\nSasuke leaves the Hidden Leaf Village
Top-5 result overlap: 5/5
Same ranking: True
\nGaara uses his sand powers
Top-5 result overlap: 5/5
Same ranking: True
\nNaruto battles a dangerous enemy
Top-5 result overlap: 5/5
Same ranking: True
\nthe 